In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from capstone.config import FEATURES, NUMERIC, PLOT, STYLES, FrankensteinConfig, LossWeights
from capstone.data import load_recipes, make_frankensteins
from capstone.models import adjust, adjust_many, blend_centroid, build_search_space, centroids, estimate_mix, fit_centroids, styles, target_distance
from capstone.utilities import setup_logging

setup_logging("WARNING")

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110
COLORS = PLOT.style_colors
SEED = 0

In [ ]:
df = load_recipes()
model = fit_centroids(df[FEATURES], df["style"])
space = build_search_space(model, df)
frank = make_frankensteins(df, FrankensteinConfig(n=300, seed=SEED))

In [ ]:
# Sanity check of the mix estimator: real recipes, and the blended centroids themselves
est_real = estimate_mix(model, df)
display(est_real.groupby(df["style"].to_numpy()).mean())

requests = [{"neapolitan": 1}, {"new-york": 1}, {"sicilian": 1},
            {"neapolitan": .5, "new-york": .5}, {"neapolitan": .3, "new-york": .7}, {"neapolitan": .7, "new-york": .3},
            {"sicilian": .5, "new-york": .5}, {"neapolitan": .5, "sicilian": .5},
            {"neapolitan": 1/3, "new-york": 1/3, "sicilian": 1/3}, {"neapolitan": .6, "new-york": .3, "sicilian": .1}]
label = lambda r: " / ".join(f"{int(round(100 * v / sum(r.values())))}% {k}" for k, v in r.items())

rows = []
for r in requests:
    centroid = blend_centroid(model, r)
    z = pd.DataFrame([centroid.to_numpy()], columns=centroid.index)
    # reconstruct the blended centroid with the same NNLS used for adjusted recipes
    from scipy.optimize import nnls
    C = centroids(model).to_numpy().T
    w = nnls(np.vstack([C, 1e3 * np.ones(3)]), np.append(centroid.to_numpy(), 1e3))[0]
    rows.append({"request": label(r), **dict(zip(styles(model), (w / w.sum()).round(3)))})
pd.DataFrame(rows).set_index("request")

In [ ]:
# Adjust the frankensteins toward each requested blend
results = {}
for r in requests:
    adjusted, info = adjust_many(frank, r, space)
    results[label(r)] = {"request": r, "adjusted": adjusted, "info": info}

def requested_vector(r):
    total = sum(r.values())
    return pd.Series({s: r.get(s, 0) / total for s in styles(model)})

rows = []
for name, res in results.items():
    r = res["request"]
    req = requested_vector(r)
    before_mix, after_mix = estimate_mix(model, frank), estimate_mix(model, res["adjusted"])
    rows.append({
        "request": name,
        "dist before": target_distance(model, frank, r).mean(),
        "dist after": target_distance(model, res["adjusted"], r).mean(),
        "mix error before": 0.5 * (before_mix - req).abs().sum(axis=1).mean(),
        "mix error after": 0.5 * (after_mix - req).abs().sum(axis=1).mean(),
        **{f"achieved {s}": after_mix[s].mean() for s in STYLES},
        "type changes": res["info"]["n_type_changes"].mean(),
        "removed": res["info"]["n_removed"].mean(),
        "added": res["info"]["n_added"].mean(),
    })
summary = pd.DataFrame(rows).set_index("request")
summary

In [ ]:
# Requested vs achieved mix on the simplex
def to_xy(m):
    m = np.asarray(m)
    return np.c_[m[:, 1] + 0.5 * m[:, 2], (np.sqrt(3) / 2) * m[:, 2]]  # neapolitan=(0,0), new-york=(1,0), sicilian=(.5,.87)

fig, ax = plt.subplots(figsize=(7, 6.2))
corners = to_xy(np.eye(3))
ax.plot(*np.vstack([corners, corners[:1]]).T, color="k", lw=1)
for name, xy in zip(styles(model), corners):
    ax.annotate(name, xy, ha="center", va="bottom" if name == "sicilian" else "top", fontsize=10)
for i, (name, res) in enumerate(results.items()):
    req = to_xy(requested_vector(res["request"]).to_numpy()[None])[0]
    ach = to_xy(estimate_mix(model, res["adjusted"]).to_numpy())
    ax.scatter(*ach.T, s=6, alpha=0.25, color=f"C{i}")
    ax.scatter(*ach.mean(axis=0), s=60, marker="o", color=f"C{i}", edgecolor="k", zorder=3)
    ax.scatter(*req, s=160, marker="*", color=f"C{i}", edgecolor="k", zorder=4)
    ax.plot(*np.c_[req, ach.mean(axis=0)], color=f"C{i}", lw=0.8, ls="--")
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("star = requested mix, dot = mean achieved mix")
fig.tight_layout()

In [ ]:
# Dose-response: slide from one style toward another in 10% steps
def dose_response(src, dst, frank, steps=np.linspace(0, 1, 11)):
    rows = []
    for t in steps:
        r = {src: 1 - t, dst: t}
        adjusted, info = adjust_many(frank, r, space)
        mix = estimate_mix(model, adjusted)
        rows.append({
            "requested": t, "achieved": mix[dst].mean(), "achieved_sd": mix[dst].std(),
            "dist_after": target_distance(model, adjusted, r).mean(),
            "type_changes": info["n_type_changes"].mean(),
            "removed": info["n_removed"].mean(), "added": info["n_added"].mean(),
            "ratio_loss": info["loss_ratio"].mean(),
        })
    return pd.DataFrame(rows).set_index("requested")

sub = frank.iloc[:150]
paths = {("new-york", "neapolitan"): None, ("neapolitan", "sicilian"): None, ("sicilian", "new-york"): None}
for (src, dst) in paths:
    paths[(src, dst)] = dose_response(src, dst, sub)
    print(f"{src} -> {dst}")
    display(paths[(src, dst)])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for col, ((src, dst), res) in enumerate(paths.items()):
    ax = axes[0, col]
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.errorbar(res.index, res["achieved"], yerr=res["achieved_sd"], marker="o", capsize=2)
    ax.set_xlabel(f"requested share of {dst}")
    ax.set_ylabel(f"achieved share of {dst}")
    ax.set_title(f"{src} -> {dst}")
    ax = axes[1, col]
    for k in ["type_changes", "removed", "added"]:
        ax.plot(res.index, res[k], marker="o", label=k)
    ax.plot(res.index, res["ratio_loss"], marker="s", ls="--", label="ratio loss")
    ax.set_xlabel(f"requested share of {dst}")
    ax.legend()
fig.tight_layout()

In [ ]:
# A real New York recipe nudged toward Neapolitan by increasing amounts, starting from its own formulation
ny = df[df["style"] == "new-york"].iloc[0]
rows, adjusted_rows = {}, {}
for t in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    r = {"new-york": 1 - t, "neapolitan": t}
    out = adjust(ny, r, space)
    adjusted_rows[f"{int(t * 100)}% neapolitan"] = out.recipe
    rows[f"{int(t * 100)}% neapolitan"] = {
        "dist to target": float(target_distance(model, out.recipe.to_frame().T, r).iloc[0]),
        "type changes": out.n_type_changes, "removed": out.n_removed, "added": out.n_added,
        "total loss": out.loss.total,
    }
print(ny.name)
display(pd.concat([ny[FEATURES].rename("original"), pd.DataFrame(adjusted_rows)], axis=1))
display(pd.DataFrame(rows).T)
display(estimate_mix(model, pd.DataFrame(adjusted_rows).T))

In [ ]:
# Does a mid-blend target reach the requested mix as strictness varies?
mid = {"new-york": 0.5, "neapolitan": 0.5}
rows = []
for name, w in {
    "default": LossWeights(),
    "lenient ratios": LossWeights(ratio=0.01),
    "cheap type swaps": LossWeights(type_change=0.25),
    "strict types": LossWeights(type_change=5.0),
    "allow adds": LossWeights(add=0.5),
    "strict ratios": LossWeights(ratio=1.0),
}.items():
    a, info = adjust_many(sub, mid, space, w)
    mix = estimate_mix(model, a)
    rows.append({
        "weights": name,
        "dist after": target_distance(model, a, mid).mean(),
        "mix error": 0.5 * (mix - requested_vector(mid)).abs().sum(axis=1).mean(),
        "achieved neapolitan": mix["neapolitan"].mean(), "achieved new-york": mix["new-york"].mean(),
        "type changes": info["n_type_changes"].mean(), "added": info["n_added"].mean(),
        "removed": info["n_removed"].mean(), "ratio loss": info["loss_ratio"].mean(),
    })
pd.DataFrame(rows).set_index("weights")

In [ ]:
# Bootstrap over frankensteins: uncertainty on mix error for each request
rng = np.random.default_rng(SEED)
rows = []
for name, res in results.items():
    err = (0.5 * (estimate_mix(model, res["adjusted"]) - requested_vector(res["request"])).abs().sum(axis=1)).to_numpy()
    means = [err[rng.integers(0, len(err), len(err))].mean() for _ in range(1000)]
    rows.append({"request": name, "mix_error": err.mean(), "ci_lo": np.quantile(means, 0.025), "ci_hi": np.quantile(means, 0.975)})
pd.DataFrame(rows).set_index("request")